# PyBaMM SEI Formulations Sensitivity Analysis
This notebook simulates calendar ageing followed by fast cycling. It sweeps through the 4 available SEI mechanism models in PyBaMM.

In [ ]:
!pip install pybamm==26.3 pandas matplotlib

In [ ]:
import pybamm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pybamm.set_logging_level("INFO")

## Base Configuration
We set up an identical `options` dictionary for a DFN model but sweep the `SEI` setting.

In [ ]:
STORAGE_DAYS = 30
INITIAL_SOC = 0.9
TOTAL_CYCLES = 100

SEI_MODELS = [
    "reaction limited",
    "solvent-diffusion limited",
    "electron-migration limited",
    "interstitial-diffusion limited"
]

EXPERIMENT_STEP = (
    "Discharge at C/9 until 3.2 V",
    "Rest for 15 minutes",
    "Charge at C/2 until 4.1 V",
    "Hold at 4.1 V until C/20",
    "Rest for 15 minutes",
)

def get_submesh_types():
    model = pybamm.lithium_ion.DFN()
    return model.default_submesh_types.copy()

var_pts = {"x_n": 30, "x_s": 30, "x_p": 30, "r_n": 50, "r_p": 50}

## Main Execution Framework
Here we loop over the 4 mechanisms, aging the cell for 30 days and then cycling it. Colab handles the full cycle set in memory at once.

In [ ]:
results_data = {}

for sei_model in SEI_MODELS:
    print(f"\n{'='*50}")
    print(f"Running SEI Model: {sei_model}")
    print(f"{'='*50}\n")
    
    options = {
        "SEI": sei_model,
        "SEI porosity change": "true",
        "lithium plating": "partially reversible",
        "lithium plating porosity change": "true",
        "particle mechanics": ("swelling and cracking", "swelling only"),
        "SEI on cracks": "true",
        "loss of active material": "stress-driven",
    }
    
    model = pybamm.lithium_ion.DFN(options)
    parameter_values = pybamm.ParameterValues("OKane2022")
    parameter_values["Current function [A]"] = 0
    solver = pybamm.IDAKLUSolver(atol=1e-6, rtol=1e-6)
    
    # Ageing
    print(f"-> Storage Phase ({STORAGE_DAYS} days)")
    seconds = STORAGE_DAYS * 24 * 60 * 60
    t_eval = np.linspace(0, seconds, min(100, max(20, STORAGE_DAYS * 2)))
    sim = pybamm.Simulation(
        model, parameter_values=parameter_values, solver=solver, var_pts=var_pts, submesh_types=get_submesh_types()
    )
    sol_ageing = sim.solve(t_eval=t_eval, initial_soc=INITIAL_SOC)
    
    # Cycling
    print(f"-> Cycling Phase ({TOTAL_CYCLES} cycles)")
    experiment = pybamm.Experiment([EXPERIMENT_STEP] * TOTAL_CYCLES)
    
    # Update current for experiment
    parameter_values_cyc = pybamm.ParameterValues("OKane2022")
    
    sim_cyc = pybamm.Simulation(
        model,
        experiment=experiment,
        parameter_values=parameter_values_cyc,
        solver=solver,
        var_pts=var_pts,
        submesh_types=get_submesh_types(),
    )
    
    cycling_solution = sim_cyc.solve(starting_solution=sol_ageing)
    
    cc_caps, cv_caps, dis_caps, cycles = [], [], [], []
    
    # Parse data from each cycle
    for cycle_sol in cycling_solution.cycles:
        steps = cycle_sol.steps
        step_dis = steps[0] if len(steps) > 0 else None   # Discharge
        step_cc  = steps[2] if len(steps) > 2 else None   # CC Charge
        step_cv  = steps[3] if len(steps) > 3 else None   # CV Hold
        
        d_cap = (
            step_dis["Discharge capacity [A.h]"].entries[-1]
            - step_dis["Discharge capacity [A.h]"].entries[0]
        ) if step_dis else 0.0
        
        c_cap = abs(
            step_cc["Discharge capacity [A.h]"].entries[-1]
            - step_cc["Discharge capacity [A.h]"].entries[0]
        ) if step_cc else 0.0
        
        v_cap = abs(
            step_cv["Discharge capacity [A.h]"].entries[-1]
            - step_cv["Discharge capacity [A.h]"].entries[0]
        ) if step_cv else 0.0
        
        dis_caps.append(d_cap)
        cc_caps.append(c_cap)
        cv_caps.append(v_cap)
        cycles.append(len(cycles) + 1)
        
    print("Done.")

    results_data[sei_model] = {
        "cycles": cycles,
        "cc_caps": cc_caps,
        "cv_caps": cv_caps,
        "dis_caps": dis_caps
    }

## Analysis and Ratios

In [ ]:
# Build Pandas DF
rows = []
for model_name, data in results_data.items():
    for c, cc, cv in zip(data["cycles"], data["cc_caps"], data["cv_caps"]):
        if c in [10, 50, 100, 150, 200]:
            rows.append({
                "Variant": model_name,
                "Cycle": c,
                "CC Capacity [A.h]": cc,
                "CV Capacity [A.h]": cv,
                "CC/CV Ratio": cc / cv if cv > 0 else float('nan')
            })

df = pd.DataFrame(rows)
if not df.empty:
    pivot_cc = df.pivot(index='Cycle', columns='Variant', values='CC Capacity [A.h]')
    pivot_cv = df.pivot(index='Cycle', columns='Variant', values='CV Capacity [A.h]')
    pivot_ratio = df.pivot(index='Cycle', columns='Variant', values='CC/CV Ratio')

    print("--- CC Capacity [A.h] ---")
    print(pivot_cc.round(3))
    print("\n--- CV Capacity [A.h] ---")
    print(pivot_cv.round(3))
    print("\n--- CC/CV Ratio ---")
    print(pivot_ratio.round(2))
else:
    print("No data at milestone cycles yet.")

## Visual Plots

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 6))

for model_name, data in results_data.items():
    axs[0].plot(data["cycles"], data["cc_caps"], marker=".", label=model_name)
    axs[1].plot(data["cycles"], data["cv_caps"], marker=".", label=model_name)
    ratio = [cc / cv if cv > 0 else 0 for cc, cv in zip(data["cc_caps"], data["cv_caps"])]
    axs[2].plot(data["cycles"], ratio, marker=".", label=model_name)

axs[0].set_title("CC Capacity Decay")
axs[0].set_ylabel("Capacity [A.h]")
axs[0].set_xlabel("Cycles")
axs[0].grid(True)
axs[0].legend()

axs[1].set_title("CV Capacity Reliance")
axs[1].set_ylabel("Capacity [A.h]")
axs[1].set_xlabel("Cycles")
axs[1].grid(True)
axs[1].legend()

axs[2].set_title("CC/CV Ratio")
axs[2].set_ylabel("Ratio")
axs[2].set_xlabel("Cycles")
axs[2].grid(True)
axs[2].legend()

plt.tight_layout()
plt.show()